# Implementing an LR-Table-Generator

## A Grammar for Grammars

We use the following grammar to describe the syntax of a context free grammar.
Since <span style="font-variant:small-caps;">Larkita</span> supports *EBNF*, we can use the postfix operators
`*` (zero or more repetitions) and `+` (one or more repetitions) and we can group symbols with parentheses.
This way, the grammar is considerably shorter than it would be in plain *BNF*:
```
grammar : rule+
rule    : RULE_NAME ":" body ("|" body)*
body    : item*
item    : VARIABLE | TOKEN | LITERAL
```
Here, `RULE_NAME`, `VARIABLE`, `TOKEN`, and `LITERAL` are *terminals*:
- a `VARIABLE` is a *syntactical variable* and is written in lower case letters,
- a `RULE_NAME` is a variable that is followed by a colon, i.e. the variable that is defined by a rule,
- a `TOKEN` is the name of a token and is written in upper case letters, while
- a `LITERAL` is a string enclosed in double quotes.

This is also the notation of the grammars in the directory `Examples`.  A rule is not terminated by a semicolon:
It ends where the next rule begins, i.e. at the next variable that is followed by a colon.

As the goal is to generate an *LR-table-generator* we first need to implement a parser for context free grammars.
The file `arith.g` in the directory `Examples` contains an example grammar that describes arithmetic expressions.

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml";;
cat "Examples/cc.g"

We use <span style="font-variant:small-caps;">Larkita</span> to develop a parser for
context free grammars.  In contrast to <span style="font-variant:small-caps;">Ply</span>, the scanner and the
parser are not specified via functions but rather via a single grammar that is given as a string.

The string `grammar_of_grammars` below is the *EBNF grammar* that describes context free grammars.
The following conventions of <span style="font-variant:small-caps;">Larkita</span> are important:
- Names written in lower case letters are *grammar rules*, while names written in upper case letters are
  *terminals*, i.e. they are the rules that are used by the scanner.
- Terminals are defined either as strings or as *regular expressions* that are enclosed in forward slashes.
- The parser returns the *abstract syntax tree* of a grammar as a nested tuple.  The string literals of an
  alternative form the *head* of the tuple that is returned for this alternative, while all other symbols become
  its children.  The strings `":"` and `"|"` occurring in the rule `rule` carry no information, so the
  annotation `%ignore ":", "|"` removes them from the head.  As no literal is left, the name of the rule is
  used as head:  A rule yields the tuple `('rule', variable, body_1, ..., body_n)` and a body yields the tuple
  `('body', item_1, ..., item_n)`.
- A token is represented as a pair consisting of the name of the token and its text, for example
  `('VARIABLE', 'expr')`.  As every alternative of the rule `item` consists of a single symbol, `item` creates no
  tuple of its own.  Instead, the tokens show up as immediate children of the tuple `('body', ...)`.
- The rules of a grammar are not terminated by a semicolon.  Therefore, after the body of a rule, a variable can
  either be the next item of this body or the beginning of the next rule.  Only the colon that follows the variable
  tells these cases apart, but an LR(1) parser sees only one token ahead.  Hence, the scanner recognizes a variable
  that is followed by a colon as a token of its own, `RULE_NAME`.  Its regular expression ends with the *lookahead*
  `(?=[ \t]*:)`, which checks that a colon follows without making the colon part of the token.  As both
  `RULE_NAME` and `VARIABLE` match the name of the variable, the priority `2` in `RULE_NAME.2` tells the scanner to
  prefer `RULE_NAME`.
- The directive `%ignore` lists those terminals that the scanner discards.  We ignore both white space and
  comments.  Note that the two slashes that start a comment have to be escaped in the regular expression
  `/\/\/[^\n]*/` because in a grammar the string `//` starts a comment.

In [ ]:
let grammar_of_grammars = {|
    grammar : rule+                                 // a grammar is a non-empty list of rules
    rule    : RULE_NAME ":" body ("|" body)*        %ignore ":", "|"
    body    : item*                                 // a body can be empty
    item    : VARIABLE | TOKEN | LITERAL

    RULE_NAME.2 : /[a-z][a-z0-9_]*(?=[ \t]*:)/      // a variable that is followed by a colon
    VARIABLE    : /[a-z][a-z0-9_]*/
    TOKEN       : /[A-Z][A-Z0-9_]*/
    LITERAL     : /".*?"/
    COMMENT     : /\/\/[^\n]*/

    %import common.WS
    %ignore WS, COMMENT
|}

The class `Larkita` creates both the scanner and the parser.  The keyword argument `start` specifies the start
symbol.  It could be omitted here, since the start symbol defaults to the first rule of the grammar.  `Larkita`
always generates a *canonical LR(1) parser* and writes a description of its states into the file given by
`~out_file`.  The argument ``~lexer:`Basic`` selects a scanner that works independently of the parser.  This
is not necessary for parsing, but it enables us to inspect the token stream via the function `Larkita.lex` that is used in the
function `test_scanner` below.

In [ ]:
let parser = Larkita.create ~start:"grammar" ~lexer:`Basic ~out_file:"grammar-of-grammars.out" grammar_of_grammars

Given the name of a file containing a grammar, the function `test_scanner` prints the tokens that
the scanner produces.  Every token `t` provides the attributes `t.type_`, `t.value`, `t.line`, and `t.column`.

In [ ]:
let test_scanner (file_name : string) : unit =
  let program = read_file file_name in
  print_endline program;
  (* start scanning and collect all tokens *)
  List.iter
    (fun (t : token) -> Printf.printf "line %2d, column %2d: %-10s%s\n%!" t.line t.column t.type_ t.value)
    (Larkita.lex parser program)

In [ ]:
test_scanner "Examples/cc.g"

The method `parse` of the object `parser` returns the *abstract syntax tree* as a nested tuple, which we print with
the function `Larkita.pprint`, which splits it into several lines if it is too long.  Note that the literals `":"` and `"|"` do not occur in this tuple and that there are
no tuples labelled `item`.

In [ ]:
Larkita.pprint (Larkita.parse parser (read_file "Examples/cc.g"))

The nested tuple is not a convenient representation for the generation of an *LR-table*.  Instead,
a grammar should be represented as a list of lists: the rule
$$ a \rightarrow \beta $$
is stored as the list `[a] @ 𝛽`.  The function `grammar_to_rules` performs this conversion.  A rule of the form
`v : b1 | b2 | b3` is represented by the tuple `("rule", ("RULE_NAME", "v"), body1, body2, body3)` and defines three
grammar rules.  The function creates the list `v :: β` for every body `("body", item_1, ..., item_n)` of every rule,
where `β` is the list of the texts of the items.

The type abbreviations `variable`, `token`, `symbol`, and `symbols` make the types more readable.

In [ ]:
type variable = string
type token    = string
type symbol   = string
type symbols  = symbol list

let grammar_to_rules (ast : nested) : symbols list =
  let rules =
    match ast with
    | Node (_ :: rules) -> rules
    | _ -> invalid_arg "grammar_to_rules: tuple expected"
  in
  let text (item : nested) : symbol =
    match item with
    | Node [ _; Leaf (Str text) ] -> text
    | _ -> invalid_arg "token expected"
  in
  List.concat_map
    (fun rule ->
      match rule with
      | Node (_ :: Node [ _; Leaf (Str variable) ] :: bodies) ->
          List.map
            (fun body ->
              match body with
              | Node (_ :: items) -> variable :: List.map text items
              | _ -> invalid_arg "grammar_to_rules: body expected")
            bodies
      | _ -> invalid_arg "grammar_to_rules: rule expected")
    rules

The function `parse` takes the name of a file containing a grammar and returns the list of the
grammar rules that are defined in this file.  If the file contains a syntax error, the parser raises an
exception `Larkita.Unexpected_input`.  Its field `kind` tells whether the scanner has found an unexpected character
(`Unexpected_characters`) or the parser has found an unexpected token (`Unexpected_token`).  The exception knows the line and the column of the error, while the function
`Larkita.get_context` returns the offending line together with a marker pointing at the error.

In [ ]:
let parse (file : string) : symbols list =
  let grammar = read_file file in
  display_text grammar;
  let ast =
    try Larkita.parse parser grammar
    with Larkita.Unexpected_input error as e ->
      display_text
        (Printf.sprintf "Syntax error in line %d, column %d:\n%s" error.line error.column (Larkita.get_context error grammar));
      raise e
  in
  grammar_to_rules ast

In [ ]:
let ruleList = parse "Examples/cc.g"
;;
ruleList

## The Module `GrammarRule`

The module `GrammarRule` is used to store a single grammar rule.  As we have to use grammar rules as keys in a
dictionary later, they have to be compared.  Records are compared *structurally*, so this works immediately.  The
function `to_string` turns a grammar rule into a string; it is installed as the printer of grammar rules.

In [ ]:
module GrammarRule = struct
  type t = { mVariable : variable; mBody : symbols }

  let make (variable : variable) (body : symbols) : t = { mVariable = variable; mBody = body }

  let to_string (self : t) : string = Printf.sprintf "%s → %s" self.mVariable (String.concat " " self.mBody)

  let pp ppf (self : t) = Format.pp_print_string ppf (to_string self)
end
;;
#install_printer GrammarRule.pp

The function `transform` takes the list of rules returned by `parse` and converts every rule into a grammar rule
of type `GrammarRule.t`.  The example below will clarify this structure.

In [ ]:
let transform (rules : symbols list) : GrammarRule.t list =
  List.map (fun rule -> GrammarRule.make (List.hd rule) (List.tl rule)) rules

In [ ]:
let grammar = transform ruleList
;;
grammar

Given a string `name`, which is either a *variable*, a *token*, or a *literal*, the function `is_var` checks whether
`name` is a variable.  The function can distinguish variable names from tokens and literals because variable names
consist only of lower case letters, while tokens are all uppercase and literals start with a double quote.  The
only variable that does not start with an ASCII letter is the variable `ŝ` of the augmented grammar.

In [ ]:
let is_var (name : symbol) : bool =
  match name.[0] with
  | 'a' .. 'z' -> true
  | c -> Char.code c >= 128   (* the variable ŝ *)

Given a list `rules` of grammar rules, the function `collect_variables rules` returns the set of all *variables* occuring in `rules`.

In [ ]:
let collect_variables (rules : GrammarRule.t list) : variable FSet.t =
  FSet.of_list
    (List.concat_map (fun (rule : GrammarRule.t) -> rule.mVariable :: List.filter is_var rule.mBody) rules)

Given a list `rules` of grammar rules, the function `collect_tokens rules` returns the set of all *tokens* and *literals* occuring in `rules`.

In [ ]:
let collect_tokens (rules : GrammarRule.t list) : token FSet.t =
  FSet.of_list (List.concat_map (fun (rule : GrammarRule.t) -> List.filter (fun item -> not (is_var item)) rule.mBody) rules)

## Extended Marked Rules

The module `ExtendedMarkedRule` stores a single *marked rule* of the form
$$ v \rightarrow \alpha \bullet \beta : L $$
where the *variable* $v$ is stored in the field `mVariable`, while $\alpha$ and $\beta$ are stored in the fields `mAlpha`and `mBeta` respectively.  The set of follow tokens $L$ is stored in the field `mFollow`. The fields `mAlpha` and `mBeta` contain lists of *grammar symbols*.  A *grammar symbol* is either
- a *variable*,
- a *token*, or
- a *literal*, i.e. a string enclosed in double quotes.


Later, we need to maintain sets of *marked rules* to represent *states*.  Records are compared structurally.  Since
the sets of type `FSet.t` have exactly one representation, this also works for the field `mFollow`.

In [ ]:
module ExtendedMarkedRule = struct
  type t = { mVariable : variable; mAlpha : symbols; mBeta : symbols; mFollow : token FSet.t }

  let make (variable : variable) (alpha : symbols) (beta : symbols) (follow : token FSet.t) : t =
    { mVariable = variable; mAlpha = alpha; mBeta = beta; mFollow = follow }

  let to_string (self : t) : string =
    let alphaStr  = String.concat " " self.mAlpha in
    let betaStr   = String.concat " " self.mBeta in
    let followStr =
      if FSet.cardinal self.mFollow > 1 then "{" ^ String.concat "," (FSet.to_list self.mFollow) ^ "}"
      else String.concat "," (FSet.to_list self.mFollow)
    in
    Printf.sprintf "%s → %s • %s: %s" self.mVariable alphaStr betaStr followStr

  let pp ppf (self : t) = Format.pp_print_string ppf (to_string self)
end
;;
#install_printer ExtendedMarkedRule.pp

Given an *extended marked rule* `self`, the function `is_complete` checks, whether the *extended marked rule* `self` has the form
$$ c \rightarrow \alpha\; \bullet: L,$$
i.e. it checks, whether the $\bullet$ is at the end of the grammar rule.

We *extend* the module `ExtendedMarkedRule`: the new module *includes* everything from the old module and adds the function `is_complete`.

In [ ]:
module ExtendedMarkedRule = struct
  include ExtendedMarkedRule

  let is_complete (self : t) : bool = self.mBeta = []
end

Given an *extended marked rule* `self` of the form
$$ c \rightarrow \alpha \bullet X\, \delta: L, $$
the function `symbol_after_dot` returns `Some` $X$. If there is no symbol after the $\bullet$, the method returns `None`.

In [ ]:
module ExtendedMarkedRule = struct
  include ExtendedMarkedRule

  let symbol_after_dot (self : t) : symbol option =
    match self.mBeta with
    | x :: _ -> Some x
    | [] -> None
end

Given an extended marked rule, this function returns `Some` $b$ if the symbol $b$ following the dot is a variable.  If there is no variable following the dot, the function returns `None`.

In [ ]:
module ExtendedMarkedRule = struct
  include ExtendedMarkedRule

  let next_var (self : t) : variable option =
    match self.mBeta with
    | var :: _ when is_var var -> Some var
    | _ -> None
end

The function `move_dot self` transforms an *extended marked rule*  of the form
$$ c \rightarrow \alpha \bullet X\, \beta: L $$
into an *extended marked rule* of the form
$$ c \rightarrow \alpha\, X \bullet \beta: L, $$
i.e. the $\bullet$ is moved over the next symbol.  Invocation of this method assumes that there is a symbol
following the $\bullet$.

In [ ]:
module ExtendedMarkedRule = struct
  include ExtendedMarkedRule

  let move_dot (self : t) : t =
    match self.mBeta with
    | x :: rest -> make self.mVariable (self.mAlpha @ [ x ]) rest self.mFollow
    | [] -> invalid_arg "move_dot: there is no symbol after the dot"
end

The function `to_rule self` turns the *extended marked rule* `self` into a grammar rule, i.e. the *extended marked rule*
$$ c \rightarrow \alpha \bullet \beta: L $$
is turned into the grammar rule
$$ c \rightarrow \alpha\, \beta. $$

In [ ]:
module ExtendedMarkedRule = struct
  include ExtendedMarkedRule

  let to_rule (self : t) : GrammarRule.t = GrammarRule.make self.mVariable (self.mAlpha @ self.mBeta)
end

The module `MarkedRule` is similar to the module `ExtendedMarkedRule` but does not have the *follow set*.  It is
needed by the function `to_marked_rule` below.

In [ ]:
module MarkedRule = struct
  type t = { mVariable : variable; mAlpha : symbols; mBeta : symbols }

  let make (variable : variable) (alpha : symbols) (beta : symbols) : t =
    { mVariable = variable; mAlpha = alpha; mBeta = beta }

  let to_string (self : t) : string =
    Printf.sprintf "%s → %s • %s" self.mVariable (String.concat " " self.mAlpha) (String.concat " " self.mBeta)
end

The function `to_marked_rule self` turns the *extended marked rule* `self` into a marked rule, i.e. the *extended marked rule*
$$ c \rightarrow \alpha \bullet \beta: L $$
is turned into the marked rule
$$ c \rightarrow \alpha\bullet \beta. $$

In [ ]:
module ExtendedMarkedRule = struct
  include ExtendedMarkedRule

  let to_marked_rule (self : t) : MarkedRule.t = MarkedRule.make self.mVariable self.mAlpha self.mBeta
end

Given a set of *extended marked rules* `m`, the function `combine_rules` *combines* those extended marked rules that have the same *core*:  If
$$ a \rightarrow \beta \bullet \gamma : L $$
is an extended marked rule, then its *core* is defined as the marked rule
$$ a \rightarrow \beta \bullet \gamma. $$
If $a \rightarrow \beta \bullet \gamma : L_1$ and $a \rightarrow \beta \bullet \gamma : L_2$ are two extended marked rules, then they can be *combined* into the rule
$$ a \rightarrow \beta \bullet \gamma : L_1\cup L_2 $$

In [ ]:
type state = ExtendedMarkedRule.t FSet.t

let combine_rules (m : state) : state =
  let cores = FSet.map ExtendedMarkedRule.to_marked_rule m in
  FSet.map
    (fun (core1 : MarkedRule.t) ->
      let follow =
        FSet.fold
          (fun emr2 acc -> if ExtendedMarkedRule.to_marked_rule emr2 = core1 then FSet.union acc emr2.ExtendedMarkedRule.mFollow else acc)
          m FSet.empty
      in
      ExtendedMarkedRule.make core1.mVariable core1.mAlpha core1.mBeta follow)
    cores

## LR-Table-Generation

The module `Grammar` represents a context free grammar.  Its record type `t` stores the list of the grammar rules of
the given grammar.  Each grammar rule has the form
$$ a \rightarrow \beta. $$
The start symbol is assumed to be the variable on the left hand side of the first rule. The grammar is *augmented* with the rule
$$ \widehat{s} \rightarrow s. $$
Here $s$ is the start variable of the given grammar and $\widehat{s}$ is a new variable that is the start variable of the *augmented grammar*. The symbol `$` denotes the end of input.  The non-obvious fields of the type `Grammar.t` have the following interpretation
- `mStates` is the set of all states of the *LR-parser*.  These states are sets of *extended marked rules*.
- `mStateNames` is a dictionary assigning names of the form `s0`, `s1`, $\cdots$, `sn` to the states stored in
  `mStates`.  The functions `action` and `goto` will be defined for *state names*, not for *states*, because
  otherwise the table representing these functions would become both huge and unreadable.
- `mConflicts` is a Boolean variable that will be set to `true` if the table generation discovers
  *shift/reduce conflicts* or *reduce/reduce conflicts*.

The fields that are computed later are declared as `mutable`.  An action is either `Shift s`, `Reduce r`, or `Accept`.
The function `make` creates a grammar.  It calls the function `compute_tables`, which is defined later.  Therefore, the module contains *forward references* for the functions that are defined later.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

type action = Shift of string | Reduce of GrammarRule.t | Accept

module Grammar = struct
  type t = {
    mRules       : GrammarRule.t list;
    mStart       : variable;
    mVariables   : variable FSet.t;
    mTokens      : token FSet.t;
    mutable mStates      : state FSet.t;
    mutable mStateNames  : (state, string) Dict.t;
    mutable mConflicts   : bool;
    mutable mRuleNames   : (GrammarRule.t, string) Dict.t;
    mutable mFirst       : (variable, token FSet.t) Dict.t;
    mutable mActionTable : (string * token, action) Dict.t;
    mutable mGotoTable   : (string * variable, string) Dict.t;
  }

  let compute_tables_ref       : (t -> unit) ref                    = ref not_yet_defined
  let compute_rule_names_ref   : (t -> unit) ref                    = ref not_yet_defined
  let compute_first_ref        : (t -> unit) ref                    = ref not_yet_defined
  let first_list_ref           : (t -> symbols -> token FSet.t) ref = ref not_yet_defined
  let all_states_ref           : (t -> unit) ref                    = ref not_yet_defined
  let compute_action_table_ref : (t -> unit) ref                    = ref not_yet_defined
  let compute_goto_table_ref   : (t -> unit) ref                    = ref not_yet_defined

  let make (rules : GrammarRule.t list) : t =
    let start = (List.hd rules).mVariable in
    let g = {
      mRules       = rules @ [ GrammarRule.make "ŝ" [ start ] ];   (* augmenting *)
      mStart       = start;
      mVariables   = FSet.add "ŝ" (collect_variables rules);
      mTokens      = FSet.add "$" (collect_tokens rules);
      mStates      = FSet.empty;
      mStateNames  = Dict.empty;
      mConflicts   = false;
      mRuleNames   = Dict.empty;
      mFirst       = Dict.empty;
      mActionTable = Dict.empty;
      mGotoTable   = Dict.empty;
    } in
    !compute_tables_ref g;
    g
end

Given a set of `variables`, the function `initialize_dictionary` returns a dictionary that assigns the empty set to all variables.

In [ ]:
let initialize_dictionary (variables : variable FSet.t) : (variable, token FSet.t) Dict.t =
  FSet.fold (fun a d -> Dict.add a FSet.empty d) variables Dict.empty

Given a `Grammar`, the function `compute_tables` computes
- the sets `First(v)` for every variable `v`,
- the set of all *states* of the *LR-Parser*,
- the *action table*, and
- the *goto table*.

Given a grammar `g`, the field `g.mFirst` is a dictionary such that `Dict.find a g.mFirst` is `First(a)` for all variables `a`.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_tables (self : t) : unit =
    self.mFirst <- initialize_dictionary self.mVariables;
    !compute_first_ref self;
    !compute_rule_names_ref self;
    !all_states_ref self;
    !compute_action_table_ref self;
    !compute_goto_table_ref self
end
;;
Grammar.compute_tables_ref := Grammar.compute_tables

The function `compute_rule_names` assigns a unique name to each *rule* of the grammar.  These names are used later
to represent *reduce actions* in the *action table*.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_rule_names (self : t) : unit =
    List.iteri (fun counter rule -> self.mRuleNames <- Dict.add rule ("r" ^ string_of_int counter) self.mRuleNames) self.mRules
end
;;
Grammar.compute_rule_names_ref := Grammar.compute_rule_names

The function `compute_first self` computes the sets $\texttt{First}(c)$ for all variables $c$ and stores them in the dictionary `mFirst`.  Abstractly, given a variable $c$ the function $\texttt{First}(c)$ is the set of all tokens that can start a string that is derived from $c$:
$$\texttt{First}(\texttt{c}) :=
  \Bigl\{ t \in T \Bigm| \exists \gamma \in (V \cup T)^*: \texttt{c} \Rightarrow^* t\,\gamma \Bigr\}.
$$
The definition of the function $\texttt{First}()$ is extended to strings from $(V \cup T)^*$ as follows:
- $\texttt{FirstList}(\varepsilon) = \{\}$.
- $\texttt{FirstList}(t \beta) = \{ t \}$  if $t \in T$.
- $\texttt{FirstList}(\texttt{a} \beta) = \left\{
       \begin{array}[c]{ll}
         \texttt{First}(\texttt{a}) \cup \texttt{FirstList}(\beta) & \mbox{if $\texttt{a} \Rightarrow^* \varepsilon$;} \\
         \texttt{First}(\texttt{a})                                & \mbox{otherwise.}
       \end{array}
       \right.
      $

If $\texttt{a}$ is a variable of $G$ and the rules defining $\texttt{a}$ are given as
$$\texttt{a} \rightarrow \alpha_1 \mid \cdots \mid \alpha_n, $$
then we have
$$\texttt{First}(\texttt{a}) = \bigcup\limits_{i=1}^n \texttt{FirstList}(\alpha_i). $$
The dictionary `mFirst` that stores this function is computed via a *fixed point iteration*.  The sets are printed in
the format `{x1, ..., xn}` by the function `set_to_string` from the file `../lib/nb.ml`.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_first (self : t) : unit =
    let out = Buffer.create 4096 in
    let change = ref true in
    while !change do
      change := false;
      List.iter
        (fun (rule : GrammarRule.t) ->
          let a, body = rule.mVariable, rule.mBody in
          let first_body = !first_list_ref self body in
          if not (FSet.subset first_body (Dict.find a self.mFirst)) then begin
            change := true;
            self.mFirst <- Dict.add a (FSet.union (Dict.find a self.mFirst) first_body) self.mFirst
          end)
        self.mRules
    done;
    Buffer.add_string out "First sets:\n";
    FSet.iter
      (fun v -> Printf.bprintf out "First(%s) = %s\n" v (set_to_string squote (FSet.to_list (Dict.find v self.mFirst))))
      self.mVariables;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_first_ref := Grammar.compute_first

Given a list of variables and tokens `alpha`, the function `first_list self alpha` computes the function $\texttt{FirstList}(\alpha)$ that has been defined above.  If `alpha` is *nullable*, then the result will contain the empty string $\varepsilon = \texttt{""}$.
The function `eps_union` is defined below, so it is called via a forward reference, too.

In [ ]:
let eps_union_ref : (token FSet.t -> token FSet.t -> token FSet.t) ref = ref not_yet_defined

module Grammar = struct
  include Grammar

  let rec first_list (self : t) (alpha : symbols) : token FSet.t =
    match alpha with
    | [] -> FSet.singleton ""
    | v :: r when is_var v -> !eps_union_ref (Dict.find v self.mFirst) (first_list self r)
    | t :: _ -> FSet.singleton t
end
;;
Grammar.first_list_ref := Grammar.first_list

The arguments `S` and `T` of `eps_union` are sets that contain tokens and, additionally, they might contain the empty string.

In [ ]:
let eps_union (s : token FSet.t) (t : token FSet.t) : token FSet.t =
  if FSet.mem "" s then
    if FSet.mem "" t then FSet.union s t
    else FSet.union (FSet.remove "" s) t
  else s
;;
eps_union_ref := eps_union

Given an augmented grammar $G = \langle V,T,R\cup\{\widehat{s} \rightarrow s\,\$\}, \widehat{s}\rangle$ 
and a variable $a$, the set of tokens that might follow $a$ is defined as:
$$\texttt{Follow}(a) := 
 \bigl\{ t \in \widehat{T} \,\bigm|\, \exists \beta,\gamma \in (V \cup \widehat{T})^*: 
                           \widehat{s} \Rightarrow^* \beta \,a\, t\, \gamma 
  \bigr\}.
$$
The function `compute_follow` computes the sets $\texttt{Follow}(a)$ for all variables $a$ via a *fixed-point iteration*.

If $\mathcal{M}$ is a set of *extended marked rules*, then the *closure* of $\mathcal{M}$ is the smallest set $\mathcal{K}$ such that
we have the following:
- $\mathcal{M} \subseteq \mathcal{K}$,
- If $a \rightarrow \beta \bullet c\, \delta: L$ is a *extended marked rule* from 
  $\mathcal{K}$, $c$ is a variable, and $t\in L$ and if, furthermore,
  $c \rightarrow \gamma$ is a grammar rule,
  then the marked rule $c \rightarrow \bullet \gamma: \texttt{First}(\delta\,t)$
  is an element of $\mathcal{K}$:
  $$(a \rightarrow \beta \bullet c\, \delta:L) \in \mathcal{K} 
         \;\wedge\; 
         (c \rightarrow \gamma) \in R \;\wedge\; t \in L
         \;\Rightarrow\; (c \rightarrow \bullet \gamma: \texttt{First}(\delta\,t)) \in \mathcal{K}
  $$

We define $\texttt{closure}(\mathcal{M}) := \mathcal{K}$.  The function `cmp_closure` computes this closure for a given set of *extended marked rules* via a *fixed-point iteration*.

In [ ]:
module Grammar = struct
  include Grammar

  let cmp_closure (self : t) (marked_Rules : state) : state =
    let rec loop (all_Rules : state) (new_Rules : state) : state =
      let more_Rules =
        let open FSet in
        let* rule = new_Rules in
        match ExtendedMarkedRule.next_var rule with
        | None -> empty
        | Some c ->
            let delta = List.tl rule.ExtendedMarkedRule.mBeta in
            let l = rule.ExtendedMarkedRule.mFollow in
            of_list
              (List.filter_map
                 (fun (newRule : GrammarRule.t) ->
                   if c = newRule.mVariable then begin
                     let newL = (let* t = l in first_list self (delta @ [ t ])) in
                     Some (ExtendedMarkedRule.make newRule.mVariable [] newRule.mBody newL)
                   end
                   else None)
                 self.mRules)
      in
      if FSet.subset more_Rules all_Rules then all_Rules
      else
        let new_Rules = FSet.diff more_Rules all_Rules in
        loop (FSet.union all_Rules new_Rules) new_Rules
    in
    loop marked_Rules marked_Rules
end

Given a set of *extended marked rules* $\mathcal{M}$ and a *grammar symbol* $X$, the function $\texttt{goto}(\mathcal{M}, X)$ 
is defined as follows:
$$\texttt{goto}(\mathcal{M}, X) := \texttt{closure}\Bigl( \bigl\{ 
   a \rightarrow \beta\, X \bullet \delta:L \bigm| (a \rightarrow \beta \bullet X\, \delta:L) \in \mathcal{M} 
   \bigr\} \Bigr).
$$

In [ ]:
module Grammar = struct
  include Grammar

  let goto (self : t) (marked_Rules : state) (x : symbol) : state =
    let result =
      FSet.map ExtendedMarkedRule.move_dot (FSet.filter (fun mr -> ExtendedMarkedRule.symbol_after_dot mr = Some x) marked_Rules)
    in
    combine_rules (cmp_closure self result)
end

The function `all_states` computes the set of all states of an *LR-parser*.  The function starts with the state
$$ \texttt{closure}\bigl(\{ \widehat{s} \rightarrow \bullet s : {\$} \}\bigr) $$
and then tries to compute new states by using the function `goto`.  This computation proceeds via a
*fixed-point iteration*.  Once all states have been computed, the function assigns names to these states.
This association is stored in the dictionary `mStateNames`.  While the states are computed, a dot is printed for
every new state, and the number of states is printed after every iteration.  The function `state_to_string` prints
a state as a set of extended marked rules.

In [ ]:
let state_to_string (state : state) : string = set_to_string ExtendedMarkedRule.to_string (FSet.to_list state)

module Grammar = struct
  include Grammar

  let all_states (self : t) : unit =
    let out = Buffer.create 4096 in
    let start_state = cmp_closure self (FSet.singleton (ExtendedMarkedRule.make "ŝ" [] [ self.mStart ] (FSet.singleton "$"))) in
    let start_state = combine_rules start_state in
    self.mStates <- FSet.singleton start_state;
    let rec loop (new_States : state FSet.t) =
      let more_States = ref FSet.empty in
      FSet.iter
        (fun rule_Set ->
          FSet.iter
            (fun mr ->
              if not (ExtendedMarkedRule.is_complete mr) then
                match ExtendedMarkedRule.symbol_after_dot mr with
                | Some x ->
                    let next_state = goto self rule_Set x in
                    if not (FSet.mem next_state self.mStates) && not (FSet.mem next_state !more_States) then begin
                      more_States := FSet.add next_state !more_States;
                      Buffer.add_string out "."
                    end
                | None -> ())
            rule_Set)
        new_States;
      if not (FSet.is_empty !more_States) then begin
        self.mStates <- FSet.union self.mStates !more_States;
        Printf.bprintf out "\n%d\n" (FSet.cardinal self.mStates);
        loop !more_States
      end
    in
    loop self.mStates;
    Buffer.add_string out "All LR-states:\n";
    self.mStateNames <- Dict.add start_state "s0" self.mStateNames;
    Printf.bprintf out "s0 = %s\n" (state_to_string start_state);
    let counter = ref 1 in
    FSet.iter
      (fun state ->
        self.mStateNames <- Dict.add state (Printf.sprintf "s%d" !counter) self.mStateNames;
        Printf.bprintf out "s%d = %s\n" !counter (state_to_string state);
        incr counter)
      (FSet.remove start_state self.mStates);
    display_text (Buffer.contents out)
end
;;
Grammar.all_states_ref := Grammar.all_states

The following function computes the *action table* and is defined as follows:
- If $\mathcal{M}$ contains an *extended marked rule* of the form $a \rightarrow \beta \bullet t\, \delta:L$
  then we have
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{shift}, \texttt{goto}(\mathcal{M},t) \rangle.$$
- If $\mathcal{M}$ contains an *extended marked rule* of the form $a \rightarrow \beta\, \bullet:L$ and we have
  $t \in L$, then we define
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{reduce}, a \rightarrow \beta \rangle$$
- If $\mathcal{M}$ contains the *extended marked rule* $\widehat{s} \rightarrow s \bullet:\{\$\}$, then we define 
  $$\texttt{action}(\mathcal{M},\$) := \texttt{accept}. $$
- Otherwise, we have
  $$\texttt{action}(\mathcal{M},t) := \texttt{error}. $$

The functions that compute the tables collect their output in a buffer and send it at once, so that a large
output can not overwhelm the notebook.

In [ ]:
(* the output is collected in a buffer, so there is no need to wait *)

In [ ]:
let action_to_string (action : action) : string =
  match action with
  | Shift s -> Printf.sprintf "('shift', '%s')" s
  | Reduce r -> Printf.sprintf "('reduce', %s)" (GrammarRule.to_string r)
  | Accept -> "'accept'"

module Grammar = struct
  include Grammar

  let compute_action_table (self : t) : unit =
    let out = Buffer.create 4096 in
    self.mActionTable <- Dict.empty;
    Buffer.add_string out "\nAction Table:\n";
    FSet.iter
      (fun state ->
        let stateName = Dict.find state self.mStateNames in
        let actionTable = ref Dict.empty in
        (* compute shift actions *)
        FSet.iter
          (fun token ->
            let newState = goto self state token in
            if not (FSet.is_empty newState) then begin
              let newName = Dict.find newState self.mStateNames in
              actionTable := Dict.add token (Shift newName) !actionTable;
              self.mActionTable <- Dict.add (stateName, token) (Shift newName) self.mActionTable;
              Printf.bprintf out "action(\"%s\", %s) = (\"shift\", %s)\n" stateName token newName
            end)
          self.mTokens;
        (* compute reduce actions *)
        FSet.iter
          (fun mr ->
            if ExtendedMarkedRule.is_complete mr then
              FSet.iter
                (fun token ->
                  let action2 = Reduce (ExtendedMarkedRule.to_rule mr) in
                  match Dict.find_opt token !actionTable with
                  | None ->
                      actionTable := Dict.add token action2 !actionTable;
                      self.mActionTable <- Dict.add (stateName, token) action2 self.mActionTable;
                      Printf.bprintf out "action(\"%s\", %s) = %s\n" stateName token (action_to_string action2)
                  | Some action1 when action1 <> action2 ->
                      self.mConflicts <- true;
                      Buffer.add_string out "\n";
                      Printf.bprintf out "conflict in state %s:\n" stateName;
                      Printf.bprintf out "%s = %s\n" stateName (state_to_string state);
                      Printf.bprintf out "action(\"%s\", %s) = %s\n" stateName token (action_to_string action1);
                      Printf.bprintf out "action(\"%s\", %s) = %s\n" stateName token (action_to_string action2);
                      Buffer.add_string out "\n"
                  | Some _ -> ())
                mr.ExtendedMarkedRule.mFollow)
          state;
        FSet.iter
          (fun mr ->
            if mr = ExtendedMarkedRule.make "ŝ" [ self.mStart ] [] (FSet.singleton "$") then begin
              actionTable := Dict.add "$" Accept !actionTable;
              self.mActionTable <- Dict.add (stateName, "$") Accept self.mActionTable;
              Printf.bprintf out "action(\"%s\", $) = accept\n" stateName
            end)
          state)
      self.mStates;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_action_table_ref := Grammar.compute_action_table

The function `compute_goto_table` computes the *goto table*.

In [ ]:
module Grammar = struct
  include Grammar

  let compute_goto_table (self : t) : unit =
    let out = Buffer.create 4096 in
    self.mGotoTable <- Dict.empty;
    Buffer.add_string out "\nGoto Table:\n";
    FSet.iter
      (fun state ->
        FSet.iter
          (fun var ->
            let newState = goto self state var in
            if not (FSet.is_empty newState) then begin
              let stateName = Dict.find state self.mStateNames in
              let newName   = Dict.find newState self.mStateNames in
              self.mGotoTable <- Dict.add (stateName, var) newName self.mGotoTable;
              Printf.bprintf out "goto(%s, %s) = %s\n" stateName var newName
            end)
          self.mVariables)
      self.mStates;
    display_text (Buffer.contents out)
end
;;
Grammar.compute_goto_table_ref := Grammar.compute_goto_table

In [ ]:
let strip_quotes (t : string) : string =
  let n = String.length t in
  if n >= 2 && t.[0] = '"' && t.[n - 1] = '"' then String.sub t 1 (n - 2) else t

In [ ]:
module Grammar = struct
  include Grammar

  let dump_parse_table (self : t) (file : string) : unit =
    let ml_list l = "[" ^ String.concat "; " (List.map repr l) ^ "]" in
    Out_channel.with_open_text file (fun handle ->
        let write s = output_string handle s in
        write "(* Grammar rules: *)\n";
        List.iter
          (fun (rule : GrammarRule.t) ->
            let rule_name = Dict.find rule self.mRuleNames in
            write (Printf.sprintf "let %s = (%s, %s)\n" rule_name (repr rule.mVariable) (ml_list rule.mBody)))
          self.mRules;
        write "\n(* Action table: *)\n";
        write "let actionTable : (string * string, action) Hashtbl.t = Hashtbl.create 64;;\n";
        Dict.iter
          (fun (s, t) action ->
            let t = strip_quotes t in
            match action with
            | Reduce rule ->
                write (Printf.sprintf "actionTable.%%{%s, %s} <- Reduce %s;;\n" (repr s) (repr t) (Dict.find rule self.mRuleNames))
            | Accept -> write (Printf.sprintf "actionTable.%%{%s, %s} <- Accept;;\n" (repr s) (repr t))
            | Shift s2 -> write (Printf.sprintf "actionTable.%%{%s, %s} <- Shift %s;;\n" (repr s) (repr t) (repr s2)))
          self.mActionTable;
        write "\n(* Goto table: *)\n";
        write "let gotoTable : (string * string, string) Hashtbl.t = Hashtbl.create 64;;\n";
        Dict.iter
          (fun (s, v) state -> write (Printf.sprintf "gotoTable.%%{%s, %s} <- %s;;\n" (repr s) (repr v) (repr state)))
          self.mGotoTable;
        write "\n(* States: *)\n";
        write "let stateNames = [\n";
        Dict.iter (fun s name -> write (Printf.sprintf "  (%s, {|%s|});\n" (repr name) (state_to_string s))) self.mStateNames;
        write "]\n")
end

The function `analyse` parses a grammar, computes its tables, and writes them to the file `parse-table.ml`.  The
function `time` from the file `../lib/nb.ml` measures the time needed, like the magic command `%%time`.

In [ ]:
let analyse (file : string) : unit =
  let rules        = parse file in
  let grammarRules = transform rules in
  let grammar      = Grammar.make grammarRules in
  Grammar.dump_parse_table grammar "parse-table.ml"

In [ ]:
analyse "Examples/cc.g"

In [ ]:
analyse "Examples/arith.g"

In [ ]:
analyse "Examples/arith-ambiguous.g"

In [ ]:
analyse "Examples/postfix.g"

In [ ]:
analyse "Examples/arith-small.g"

In [ ]:
time (fun () -> analyse "Examples/c-grammar.g")